# 🎬 Movie Review Sentiment Analysis — TRAIN FILE
**Topic:** NLP + Machine Learning (Supervised Classification)  
**Dataset:** `movie_reviews.csv` — 100 reviews, 5 sentiment classes × 20 each  
**Classes:** Highly Positive | Positive | Neutral | Negative | Highly Negative  
**Pipeline:** Dataset → Pre-processing → Feature Extraction (TF-IDF) → ML Model → Evaluation → Save `.pkl`  
**Folder:** `Movie_Sentiment_Analysis/` → subfolders: `data/` | `train/` | `predict/`

In [ ]:
# ── Cell 1 — Install required libraries ──────────────────────────────────────
!pip install nltk scikit-learn pandas matplotlib seaborn -q
print('✅ Libraries installed.')

In [ ]:
# ── Cell 2 — Imports ──────────────────────────────────────────────────────────
import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet']:
    nltk.download(pkg, quiet=True)

import pandas as pd
import numpy as np
import string
import warnings
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# NLP
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# ML Models (Supervised Classification)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

print('✅ All imports done.')

In [ ]:
# ── Cell 3 — Mount Google Drive & Set Folder Paths ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Folder Structure ──
BASE_DIR    = '/content/drive/MyDrive/Movie_Sentiment_Analysis'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
PREDICT_DIR = os.path.join(BASE_DIR, 'predict')

# Create folders if they don't exist
for folder in [DATA_DIR, TRAIN_DIR, PREDICT_DIR]:
    os.makedirs(folder, exist_ok=True)

print('✅ Google Drive mounted.')
print(f'   Base    : {BASE_DIR}')
print(f'   Data    : {DATA_DIR}')
print(f'   Train   : {TRAIN_DIR}')
print(f'   Predict : {PREDICT_DIR}')

In [ ]:
# ── Cell 4 — Upload Dataset to Drive/data/ folder ─────────────────────────────
from google.colab import files
print('📂 Upload your movie_reviews.csv file now:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]

# Move uploaded file to the data/ subfolder in Drive
import shutil
dest_path = os.path.join(DATA_DIR, 'movie_reviews.csv')
shutil.copy(filename, dest_path)

# Load dataset
df = pd.read_csv(dest_path)

print(f'\n✅ Dataset loaded from: {dest_path}')
print(f'   Total rows    : {len(df)}')
print(f'   Columns       : {df.columns.tolist()}')
print(f'\n📊 Class Distribution:')
print(df['sentiment'].value_counts())
df.head()

In [ ]:
# ── Cell 5 — STEP 1: Pre-processing ──────────────────────────────────────────
# NLP Pre-processing Pipeline:
#   1. Lowercase
#   2. Remove punctuation
#   3. Tokenization
#   4. Remove stopwords
#   5. Lemmatization

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Step 1: Lowercase
    text = str(text).lower()
    # Step 2: Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    # Step 4: Remove stopwords + non-alpha
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    # Step 5: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply preprocessing to the review column
df['clean_review'] = df['review'].apply(preprocess)

print('✅ Pre-processing complete.')
print('\nSample — Original vs Cleaned:')
for i in range(3):
    print(f'\n  Original : {df["review"].iloc[i]}')
    print(f'  Cleaned  : {df["clean_review"].iloc[i]}')

In [ ]:
# ── Cell 6 — STEP 2: Feature Extraction (TF-IDF) ─────────────────────────────
# In ML, feature extraction is done manually.
# TF-IDF (Term Frequency - Inverse Document Frequency) converts text to numbers.
# TF  = how often a word appears in a review
# IDF = how rare the word is across all reviews
# Result: a numeric matrix where each row = one review, each column = one word

X = df['clean_review']   # Input  (text)
y = df['sentiment']      # Output (label)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=500,      # Top 500 most important words
    ngram_range=(1, 2),    # Unigrams + Bigrams
    sublinear_tf=True      # Apply log normalization
)

X_tfidf = tfidf.fit_transform(X)

print('✅ TF-IDF Feature Extraction complete.')
print(f'   Shape of feature matrix : {X_tfidf.shape}')
print(f'   Rows = reviews           : {X_tfidf.shape[0]}')
print(f'   Columns = TF-IDF features: {X_tfidf.shape[1]}')
print(f'\n   Sample top features      : {tfidf.get_feature_names_out()[:20].tolist()}')

In [ ]:
# ── Cell 7 — STEP 3: Train/Test Split ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,
    stratify=y           # Keep class balance in both splits
)

print('✅ Train/Test Split done.')
print(f'   Training samples : {X_train.shape[0]}')
print(f'   Testing  samples : {X_test.shape[0]}')
print(f'\n   Train class distribution:')
print(pd.Series(y_train).value_counts())

In [ ]:
# ── Cell 8 — STEP 4: Choose & Train ML Algorithms (Supervised Classification) ─
# We train 4 different supervised ML classifiers and compare them.
# These are all classification algorithms — NOT clustering.

models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes'         : MultinomialNB(),
    'Linear SVM'          : LinearSVC(max_iter=2000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = round(accuracy_score(y_test, y_pred) * 100, 2)
    prec = round(precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    rec  = round(recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    f1   = round(f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    results[name] = {'model': model, 'preds': y_pred, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
    print(f'  [{name}]  Acc={acc}%  Prec={prec}%  Rec={rec}%  F1={f1}%')

print('\n✅ All 4 models trained and evaluated.')

In [ ]:
# ── Cell 9 — STEP 5: Evaluation — Pick Best Model ────────────────────────────
best_name = max(results, key=lambda k: results[k]['Accuracy'])
best      = results[best_name]

print('='*58)
print('   MODEL COMPARISON SUMMARY')
print('='*58)
print(f'  {"Model":<22} {"Acc":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-'*58)
for name, r in results.items():
    marker = ' ← BEST' if name == best_name else ''
    print(f'  {name:<22} {r["Accuracy"]:>7}% {r["Precision"]:>7}% {r["Recall"]:>7}% {r["F1"]:>7}%{marker}')
print('='*58)
print(f'\n✅ Best Model : {best_name}')
print(f'   Accuracy   : {best["Accuracy"]}%')
print(f'   Precision  : {best["Precision"]}%')
print(f'   Recall     : {best["Recall"]}%')
print(f'   F1 Score   : {best["F1"]}%')

In [ ]:
# ── Cell 10 — Detailed Classification Report ──────────────────────────────────
print(f'📋 Classification Report — {best_name}\n')
print(classification_report(y_test, best['preds'], zero_division=0))

In [ ]:
# ── Cell 11 — Confusion Matrix ────────────────────────────────────────────────
class_labels = ['Highly Negative', 'Negative', 'Neutral', 'Positive', 'Highly Positive']

cm = confusion_matrix(y_test, best['preds'], labels=class_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {best_name}', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

# Metrics Bar Chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
values  = [best['Accuracy'], best['Precision'], best['Recall'], best['F1']]
colors  = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = axes[1].bar(metrics, values, color=colors, alpha=0.87)
for bar, val in zip(bars, values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=11
    )
axes[1].set_ylim(0, 115)
axes[1].set_title(f'Evaluation Metrics — {best_name}', fontsize=11)
axes[1].set_ylabel('Score %')
axes[1].axhline(80, color='gray', linestyle='--', linewidth=1, label='80% baseline')
axes[1].legend(fontsize=9)

plt.tight_layout()
plot_path = os.path.join(TRAIN_DIR, 'evaluation_metrics.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'✅ Plot saved to: {plot_path}')

In [ ]:
# ── Cell 12 — Model Comparison Bar Chart ──────────────────────────────────────
model_names = list(results.keys())
accs  = [results[m]['Accuracy']  for m in model_names]
f1s   = [results[m]['F1']        for m in model_names]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#4C72B0', alpha=0.87)
b2 = ax.bar(x + w/2, f1s,  w, label='F1 Score', color='#55A868', alpha=0.87)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('ML Model Comparison — Accuracy vs F1 Score', fontsize=13)
ax.set_ylabel('Score %')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 115)
ax.axhline(80, color='gray', linestyle='--', linewidth=1, label='80% baseline')
ax.legend(fontsize=9)
plt.tight_layout()
comp_path = os.path.join(TRAIN_DIR, 'model_comparison.png')
plt.savefig(comp_path, dpi=150)
plt.show()
print(f'✅ Comparison chart saved to: {comp_path}')

In [ ]:
# ── Cell 13 — Save Best Model + TF-IDF Vectorizer as .pkl ─────────────────────
# We save BOTH the trained model AND the vectorizer.
# The vectorizer must be saved too — it transforms text the same way during prediction.

model_path  = os.path.join(TRAIN_DIR, 'sentiment_model.pkl')
tfidf_path  = os.path.join(TRAIN_DIR, 'tfidf_vectorizer.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(best['model'], f)

with open(tfidf_path, 'wb') as f:
    pickle.dump(tfidf, f)

print('✅ Model and Vectorizer saved to Google Drive!')
print(f'   Model path    : {model_path}')
print(f'   Vectorizer    : {tfidf_path}')
print(f'   Best Algorithm: {best_name}')
print(f'   Accuracy      : {best["Accuracy"]}%')
print(f'   F1 Score      : {best["F1"]}%')
print('\n🎯 Training complete. Open predict.ipynb to make predictions!')